In [1]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder, OneHotEncoder
import pickle

In [2]:
data=pd.read_csv('Churn_Modelling.csv')
data.head()

,RowNumber,CustomerId,Surname,CreditScore,Geography,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Exited
0,1,15634602,Hargrave,619,France,Female,42,2,0.00,1,1,1,101348.88,1
1,2,15647311,Hill,608,Spain,Female,41,1,83807.86,1,0,1,112542.58,0
2,3,15619304,Onio,502,France,Female,42,8,159660.80,3,1,0,113931.57,1
3,4,15701354,Boni,699,France,Female,39,1,0.00,2,0,0,93826.63,0
4,5,15737888,Mitchell,850,Spain,Female,43,2,125510.82,1,1,1,79084.10,0


In [3]:
# EstimatedSalary was my output feature and remaining all columns are independant feature

# Preprocess the data
data = data.drop(['RowNumber', 'CustomerId', 'Surname'], axis=1)


In [4]:
# Encode categorical variables
label_encoder_gender = LabelEncoder()
data['Gender'] = label_encoder_gender.fit_transform(data['Gender'])

In [5]:
# One-hot encode 'Geography'
onehot_encoder_geo = OneHotEncoder(handle_unknown='ignore')
geo_encoded = onehot_encoder_geo.fit_transform(data[['Geography']]).toarray()
geo_encoded_df = pd.DataFrame(geo_encoded, columns=onehot_encoder_geo.get_feature_names_out(['Geography']))

In [6]:
geo_encoded_df

,Geography_France,Geography_Germany,Geography_Spain
0,1.0,0.0,0.0
1,0.0,0.0,1.0
2,1.0,0.0,0.0
3,1.0,0.0,0.0
4,0.0,0.0,1.0
...,...,...,...
9995,1.0,0.0,0.0
9996,1.0,0.0,0.0
9997,1.0,0.0,0.0
9998,0.0,1.0,0.0


In [7]:
# Combine one-hot encoded columns with original data
data = pd.concat([data.drop('Geography', axis=1), geo_encoded_df], axis=1)
data.head()

,CreditScore,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Exited,Geography_France,Geography_Germany,Geography_Spain
0,619,0,42,2,0.00,1,1,1,101348.88,1,1.0,0.0,0.0
1,608,0,41,1,83807.86,1,0,1,112542.58,0,0.0,0.0,1.0
2,502,0,42,8,159660.80,3,1,0,113931.57,1,1.0,0.0,0.0
3,699,0,39,1,0.00,2,0,0,93826.63,0,1.0,0.0,0.0
4,850,0,43,2,125510.82,1,1,1,79084.10,0,0.0,0.0,1.0


In [8]:
# Split the data into features and target
x = data.drop('EstimatedSalary', axis=1)
y = data['EstimatedSalary']

In [10]:
## Split the data in training and tetsing sets
x_train,x_test,y_train,y_test=train_test_split(x,y,test_size=0.2,random_state=42)

In [11]:
## Scale these features
scaler=StandardScaler()
x_train=scaler.fit_transform(x_train)
x_test=scaler.transform(x_test)

In [ ]:
# Save the encoders and scaler for later use
with open('label_encoder_gender.pkl', 'wb') as file:
    pickle.dump(label_encoder_gender, file)

with open('onehot_encoder_geo.pkl', 'wb') as file:
    pickle.dump(onehot_encoder_geo, file)

with open('scaler.pkl', 'wb') as file:
    pickle.dump(scaler, file)

##### ANN Regression Problem Statement

In [12]:
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense

In [13]:
# When we don't apply any activation function, the default activation function that will be applied is
# called as linear activation function.
# Now linear activation function is specifically for regression.

# Build the model
model = Sequential([
    Dense(64, activation='relu', input_shape=(x_train.shape[1],)),
    Dense(32, activation='relu'),
    Dense(1)
])

# compile the model
model.compile(optimizer='adam', loss='mean_absolute_error', metrics=['mae'])

model.summary()



Model: "sequential"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 dense (Dense)               (None, 64)                832       
                                                                 
 dense_1 (Dense)             (None, 32)                2080      
                                                                 
 dense_2 (Dense)             (None, 1)                 33        
                                                                 
Total params: 2945 (11.50 KB)
Trainable params: 2945 (11.50 KB)
Non-trainable params: 0 (0.00 Byte)
_________________________________________________________________


In [14]:
from tensorflow.keras.callbacks import EarlyStopping, TensorBoard
import datetime

# Set up TensorBoard
log_dir = "regressionlogs/fit/" + datetime.datetime.now().strftime("%Y%m%d-%H%M%S")
tensorboard_callback = TensorBoard(log_dir=log_dir, histogram_freq=1)

In [15]:
# Set up Early Stopping
early_stopping_callback = EarlyStopping(monitor='val_loss', patience=10, restore_best_weights=True)


In [16]:
# Train the model
history = model.fit(
    x_train, y_train,
    validation_data=(x_test, y_test),
    epochs=100,
    callbacks=[early_stopping_callback, tensorboard_callback]
)

Epoch 1/100


250/250 [==============================] - 3s 5ms/step - loss: 100387.3125 - mae: 100387.3125 - val_loss: 98551.5234 - val_mae: 98551.5234
Epoch 2/100
250/250 [==============================] - 1s 4ms/step - loss: 99753.5781 - mae: 99753.5781 - val_loss: 97265.5312 - val_mae: 97265.5312
Epoch 3/100
250/250 [==============================] - 1s 3ms/step - loss: 97503.0547 - mae: 97503.0547 - val_loss: 93950.3516 - val_mae: 93950.3516
Epoch 4/100
250/250 [==============================] - 1s 3ms/step - loss: 93005.2422 - mae: 93005.2422 - val_loss: 88299.9375 - val_mae: 88299.9375
Epoch 5/100
250/250 [==============================] - 1s 4ms/step - loss: 86298.6875 - mae: 86298.6875 - val_loss: 80745.2109 - val_mae: 80745.2109
Epoch 6/100
250/250 [==============================] - 1s 3ms/step - loss: 78057.8984 - mae: 78057.8984 - val_loss: 72376.6719 - val_mae: 72376.6719
Epoch 7/100
250/250 [==============================] - 2s 7ms/step - loss: 69570.4922 - mae: 69570.492

In [17]:
%load_ext tensorboard

In [19]:
%tensorboard --logdir regressionlogs/fit --port 6011


In [ ]:
# So let's go ahead and evaluate model on the test data. 
# We're just going to check the accuracy how good it is.
# If more near to zero. If it is going on, we can probably say that our model is pretty trained in an amazing way.
test_loss,test_mae=model.evaluate(x_test,y_test)
print(f'Test MAE : {test_mae}')


63/63 [==============================] - 0s 2ms/step - loss: 50248.2812 - mae: 50248.2812
Test MAE : 50248.28125
Test loss : 50248.28125


In [ ]:
import numpy as np

mae_percentage = (test_mae / np.mean(y_test)) * 100
print(f"MAE % of target mean: {mae_percentage:.2f}%")

MAE % of target mean: 50.90%


In [23]:
model.save('regression_model.h5')

c:\Users\Nimap\Desktop\ANNProjectImplementation\ann-project-implementation\venv\lib\site-packages\keras\src\engine\training.py:3103: UserWarning: You are saving your model as an HDF5 file via `model.save()`. This file format is considered legacy. We recommend using instead the native Keras format, e.g. `model.save('my_model.keras')`.
  saving_api.save_model(
